# CASCADE/OASIS Notebook for Suite2P traces

## Imports and Config

In [1]:
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, os.path.join("libs", "CascadeTorch"))
sys.path.insert(0, os.path.join("libs", "OASIS"))

from cascade2p import cascade
from oasis.functions import deconvolve

traces_path = os.path.join("data", "suite2p_traces", "suite2p_dff_traces.csv")
output_dir = os.path.join("data", "spike_detection")
figures_dir = "figures"


frame_rate = 30.0
model_name = "Global_EXC_30Hz_smoothing25ms"
device_name = "cpu"
max_plot_neurons = 6
label = "Suite2p"
plot_roi_indices = None

/Users/arjunphanse/calcium-firing/libs/OASIS/oasis/functions.py:13: UserWarning: Could not find cvxpy. Don't worry, you can still use OASIS, just not the slower interior point methods we compared to in the papers.
  warn("Could not find cvxpy. Don't worry, you can still use OASIS, " +


## Load dF/F traces

In [2]:
def load_traces(path):
    suffix = os.path.splitext(path)[1]
    if suffix == ".npy":
        traces = np.load(path)
    elif suffix == ".csv":
        traces = np.loadtxt(path, delimiter=",")
    else:
        raise ValueError(f"Unsupported trace format: {suffix}. Use .csv or .npy")
    return traces.astype(np.float32, copy=False)

traces = load_traces(traces_path)
os.makedirs(output_dir, exist_ok=True)
np.save(os.path.join(output_dir, "input_dff_traces.npy"), traces)

print("Trace path:", traces_path)
print("Trace shape:", traces.shape)
print("dtype:", traces.dtype)

Trace path: data/suite2p_traces/suite2p_dff_traces.csv
Trace shape: (39, 3000)
dtype: float32


## Run CASCADE

In [3]:
model_folder = os.path.join("libs", "CascadeTorch", "Pretrained_models")
device = torch.device(device_name)

cascade.download_model(model_name, model_folder=model_folder, verbose=1)

cascade_pred = cascade.predict(
    model_name,
    traces,
    model_folder=model_folder,
    device=device,
)

cascade_pred = cascade_pred.astype(np.float32)
np.save(os.path.join(output_dir, "cascade_spike_rates.npy"), cascade_pred)

print("cascade_pred shape:", cascade_pred.shape)

Pretrained model was saved in folder "/Users/arjunphanse/calcium-firing/libs/CascadeTorch/Pretrained_models/Global_EXC_30Hz_smoothing25ms"

 
The selected model was trained on 18 datasets, with 5 ensembles for each noise level, at a sampling rate of 30Hz, with a resampled ground truth that was smoothed with a Gaussian kernel of a standard deviation of 25 milliseconds. 
 

Loaded model was trained at frame rate 30 Hz
Given argument traces contains 39 neurons and 3000 frames.
Noise levels (mean, std; in standard units): 2.14, 1.73

Predictions for noise level 2:
	... ensemble 0
	... ensemble 1
	... ensemble 2
	... ensemble 3
	... ensemble 4

Predictions for noise level 3:
	... ensemble 0
	... ensemble 1
	... ensemble 2
	... ensemble 3
	... ensemble 4

Predictions for noise level 4:
	... ensemble 0
	... ensemble 1
	... ensemble 2
	... ensemble 3
	... ensemble 4

Predictions for noise level 5:
	No neurons for this noise level

Predictions for noise level 6:
	No neurons for this noise level

## Run OASIS

In [4]:
oasis_spikes = np.zeros_like(traces, dtype=np.float32)
oasis_calcium = np.zeros_like(traces, dtype=np.float32)

for neuron_idx, trace in enumerate(traces):
    calcium, spikes, _, _, _ = deconvolve(trace.astype(float), penalty=1)
    oasis_calcium[neuron_idx] = calcium
    oasis_spikes[neuron_idx] = spikes

np.save(os.path.join(output_dir, "oasis_spikes.npy"), oasis_spikes)
np.save(os.path.join(output_dir, "oasis_denoised_calcium.npy"), oasis_calcium)

print("oasis_spikes shape:", oasis_spikes.shape)

oasis_spikes shape: (39, 3000)


## Visualize dF/F, OASIS, CASCADE outputs for selected neurons

In [5]:
def normalize_for_plot(values):
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        return np.zeros_like(values)
    lo, hi = np.nanpercentile(values[finite], [1, 99])
    if hi <= lo:
        return np.zeros_like(values)
    return np.clip((values - lo) / (hi - lo), 0, 1)

noise = np.nanmedian(np.abs(np.diff(traces, axis=1)), axis=1) / 0.6745 / np.sqrt(2)
baseline = np.nanpercentile(traces, 50, axis=1)
peak = np.nanpercentile(traces, 99, axis=1)
roi_scores = (peak - baseline) / (noise + 1e-6)

roi_indices = np.argsort(roi_scores)[::-1][:5]

print("plot ROI indices:", roi_indices.tolist())

plot ROI indices: [2, 3, 6, 9, 25]


## Save comparison plots

In [7]:
os.makedirs(figures_dir, exist_ok=True)

n_neurons = len(roi_indices)
n_frames = min(traces.shape[1], round(frame_rate * 30))
time_axis = np.arange(n_frames) / frame_rate

fig, axes = plt.subplots(n_neurons, 1, figsize=(12, max(3, 1.8 * n_neurons)), sharex=True)
axes = np.atleast_1d(axes)
for plot_idx, ax in enumerate(axes):
    roi_idx = roi_indices[plot_idx]
    ax.plot(time_axis, traces[roi_idx, :n_frames], color="#555555", lw=0.9)
    ax.set_ylabel(f"ROI {roi_idx}")
axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"Selected {label} dF/F traces")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "input_traces.png"), dpi=200)
plt.close(fig)

fig, axes = plt.subplots(n_neurons, 1, figsize=(12, max(3, 1.8 * n_neurons)), sharex=True)
axes = np.atleast_1d(axes)
for plot_idx, ax in enumerate(axes):
    roi_idx = roi_indices[plot_idx]
    ax.plot(time_axis, normalize_for_plot(traces[roi_idx, :n_frames]), color="#777777", lw=0.8)
    ax.plot(time_axis, normalize_for_plot(cascade_pred[roi_idx, :n_frames]), color="#2468a2", lw=0.9)
    ax.plot(time_axis, normalize_for_plot(oasis_spikes[roi_idx, :n_frames]), color="#19945f", lw=0.9, alpha=0.85)
    ax.set_ylabel(f"ROI {roi_idx}")
axes[-1].set_xlabel("Time (s)")
axes[0].legend(["dF/F", "CASCADE", "OASIS"], loc="upper right")
fig.suptitle(f"CASCADE and OASIS on selected {label} traces")
fig.tight_layout()
fig.savefig(os.path.join(figures_dir, "cascade_oasis_traces.png"), dpi=200)
plt.close(fig)